In [ ]:
import fitz          # PyMuPDF
import unicodedata
import re
import pandas as pd

def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")

def extract_blocks(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    full = ""
    for page in doc:
        full += page.get_text("text") + "\n\f\n"
    full = strip_accents(full)
    return full.split("FICHA TECNICA")[1:]

def parse_block(block: str) -> dict | None:
    lines = [ln.strip() for ln in strip_accents(block).splitlines() if ln.strip()]
    upper = [ln.upper() for ln in lines]

    # --- (tu lógica de salto de encabezados) ---
    headers = ["PROYECTO","RADICACION","TIPO DE PROYECTO","SEUDONIMO"]
    if len(upper)>=4 and all(upper[i]==headers[i] for i in range(4)):
        lines = lines[4:]; upper = upper[4:]

    rec = {}

    # — No. CÁMARA con posible “ACU AL”
    if "NO. CAMARA" in upper:
        i = upper.index("NO. CAMARA")
        if i+1 < len(lines):
            cam_line = lines[i+1]
            if "ACU AL" in cam_line.upper():
                left, right = re.split(r"ACU AL", cam_line, flags=re.IGNORECASE)
                m1 = re.search(r"(\d+)/\d{4}[CS]", left)
                m2 = re.search(r"(\d+)/\d{4}[CS]", right)
                if m1 and m2:
                    rec["num_camara"] = m1.group(1)
                    rec["acu_al"]     = m2.group(1)
            else:
                m = re.search(r"(\d+)/\d{4}[CS]", cam_line)
                if m:
                    rec["num_camara"] = m.group(1)

    # — No. SENADO (si existe)
    if "NO. SENADO" in upper:
        j = upper.index("NO. SENADO")
        if j+1 < len(lines):
            sen_line = lines[j+1]
            m = re.search(r"(\d+)/\d{4}[CS]", sen_line)
            if m:
                rec["num_senado"] = m.group(1)

    # — Fecha de radicación
    fecha = next((ln for ln in lines if re.fullmatch(r"\d{2}/\d{2}/\d{4}", ln)), None)
    if not fecha:
        return None
    rec["fecha_radicacion"] = fecha

    # — Tipo proyecto y seudónimo
    idx_f = lines.index(fecha)
    if idx_f+1 < len(lines): rec["tipo_proyecto"] = lines[idx_f+1]
    if idx_f+2 < len(lines): rec["seudonimo"]     = lines[idx_f+2]

    # — Comisión y Cámara de origen
    if "COMISION" in upper:
        k = upper.index("COMISION")
        if k+1 < len(lines): rec["comision"] = lines[k+1]
    if "CAMARA DE ORIGEN" in upper:
        k = upper.index("CAMARA DE ORIGEN")
        if k+1 < len(lines): rec["camara_origen"] = lines[k+1]

    # — Título
    if "TITULO" in upper:
        t0 = upper.index("TITULO")
        buf = []
        for ln in lines[t0+1:]:
            if ln.upper().startswith(("AUTOR","PRIMERA","SEGUNDA","MIEMBROS","PUBLICACIONES","ESTADO")):
                break
            buf.append(ln)
        rec["titulo"] = " ".join(buf)

    # — Autores
    autor_idx = next((idx for idx,up in enumerate(upper) if up.startswith("AUTOR")), None)
    if autor_idx is not None:
        buf = []
        for ln in lines[autor_idx+1:]:
            if ln.upper().startswith(("PRIMERA","SEGUNDA","PUBLICACIONES","ESTADO","PONENTES","MIEMBROS")):
                break
            buf.append(ln)
        rec["autores"] = " ".join(buf)

    # — NUEVO: Estado actual (dato vertical al final de la ficha)
    estado_idx = next((i for i,up in enumerate(upper) if up.startswith("ESTADO")), None)
    if estado_idx is not None:
        # Si está en la misma línea tras “:”
        if ":" in lines[estado_idx]:
            rec["estado_actual"] = lines[estado_idx].split(":",1)[1].strip()
        # Si está en la siguiente línea
        elif estado_idx+1 < len(lines):
            rec["estado_actual"] = lines[estado_idx+1].strip()

    return rec


# ——— EJECUCIÓN ———

pdf_path = r"C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2020 2021 LEGISLATURA_fichas.pdf"
out_xlsx = "fichas_tecnicas_con_estado_2020_2021.xlsx"

bloques    = extract_blocks(pdf_path)
parsed     = [parse_block(b) for b in bloques]
registros  = [r for r in parsed if r]

df = pd.DataFrame(registros)
print(f"✅ Fichas válidas extraídas: {len(df)}")
display(df.head())

# Asegúrate de incluir 'estado_actual' en tus columnas si quieres controlar el orden:
cols = [
    "num_camara","acu_al","num_senado","fecha_radicacion",
    "tipo_proyecto","seudonimo","comision","camara_origen",
    "titulo","autores","estado_actual"
]
df = df.loc[:, [c for c in cols if c in df.columns]]

df.to_excel(out_xlsx, index=False)
print("✅ Excel guardado en", out_xlsx)


✅ Fichas válidas extraídas: 0


""


✅ Excel guardado en fichas_tecnicas_con_estado_2020_2021.xlsx


In [6]:
import re
import pdfplumber
import easyocr
import pandas as pd
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Inicializa EasyOCR en GPU
reader = easyocr.Reader(['es'], gpu=True)

# Mapea meses a número
month_map = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04',
    'mayo':'05','junio':'06','julio':'07','agosto':'08',
    'septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

def parse_date_spanish(date_str: str) -> str:
    # Recibe "20 de Julio de 2020" (o minúsculas), devuelve "20/07/2020"
    parts = date_str.lower().split()
    if len(parts) >= 5 and parts[1]=='de' and parts[3]=='de':
        day = parts[0].zfill(2)
        mon = month_map.get(parts[2], '00')
        year = parts[4]
        return f"{day}/{mon}/{year}"
    return date_str

def extract_text(path: str) -> str:
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text()
            if txt and txt.strip():
                pages.append(txt)
            else:
                img = page.to_image(resolution=300).original
                ocr = reader.readtext(img, detail=0)
                pages.append("\n".join(ocr))
    # Unimos con 2 saltos (no \f)
    return "\n\n".join(pages)

# 1) Carga y OCR/fallback
pdf_path = '2020 2021 LEGISLATURA_fichas.pdf'
raw = extract_text(pdf_path)

# 2) Separa cada ficha por su encabezado
bloques = re.split(r"Proyecto\s+Cámara No\.", raw, flags=re.IGNORECASE)[1:]

rows = []
for bloq in bloques:
    text = "Proyecto Cámara No." + bloq  # lo devolvemos para que la regex encuentre el encabezado

    # Cámara No. → "001/2020C" → "001"
    m = re.search(r"Cámara No\.\s*([\d/CS]+)", text, flags=re.IGNORECASE)
    cam_no = m.group(1).split('/')[0].strip() if m else ''

    # Senado No. → idem
    m = re.search(r"Senado No\.\s*([\d/CS]+)", text, flags=re.IGNORECASE)
    sen_no = m.group(1).split('/')[0].strip() if m else ''

    # Seudónimo (multilínea)
    m = re.search(r"Seudónimo:\s*(.*?)\s*Tipo de Ley:", text, flags=re.S|re.IGNORECASE)
    seud = m.group(1).replace('\n',' ').strip() if m else ''

    # Tipo de Ley
    m = re.search(r"Tipo de Ley:\s*(.*?)\s*Fecha de Radicación", text, flags=re.S|re.IGNORECASE)
    tipl = m.group(1).strip() if m else ''

    # Fecha Rad. Cámara → parse
    m = re.search(r"Fecha de Radicación\s*Cámara:\s*([0-9]{1,2}\s+de\s+\w+\s+de\s+\d{4})", text, flags=re.IGNORECASE)
    fcam = parse_date_spanish(m.group(1)) if m else ''

    # Fecha Rad. Senado → parse
    m = re.search(r"Senado:\s*([0-9]{1,2}\s+de\s+\w+\s+de\s+\d{4})", text, flags=re.IGNORECASE)
    fsen = parse_date_spanish(m.group(1)) if m else ''

    # Origen (multilínea)
    m = re.search(r"Origen:\s*(.*?)\s*Comisión:", text, flags=re.S|re.IGNORECASE)
    ori = m.group(1).replace('\n',' ').strip() if m else ''

    # Comisión (multilínea)
    m = re.search(r"Comisión:\s*(.*?)\s*Título:", text, flags=re.S|re.IGNORECASE)
    com = m.group(1).replace('\n',' ').strip() if m else ''

    # Título (multilínea)
    m = re.search(r"Título:\s*(.*?)\s*Autor\(es\):", text, flags=re.S|re.IGNORECASE)
    tit = m.group(1).replace('\n',' ').strip() if m else ''

    # Autor(es) (multilínea)
    m = re.search(r"Autor\(es\):\s*(.*?)\s*Estado Actual:", text, flags=re.S|re.IGNORECASE)
    aut = m.group(1).replace('\n',' ').strip() if m else ''

    # Estado Actual → hasta doble salto o fin de bloque
    m = re.search(r"Estado Actual:\s*([\s\S]*?)(?:\n{2,}|$)", text, flags=re.IGNORECASE)
    est = m.group(1).replace('\n',' ').strip() if m else ''

    rows.append({
        'Cámara No.':         cam_no,
        'Senado No.':         sen_no,
        'Seudónimo':          seud,
        'Tipo de Ley':        tipl,
        'Fecha Rad. Cámara':  fcam,
        'Fecha Rad. Senado':  fsen,
        'Origen':             ori,
        'Comisión':           com,
        'Título':             tit,
        'Autor(es)':          aut,
        'Estado Actual':      est,
    })

# 3) DataFrame y limpieza de caracteres ilegales
df = pd.DataFrame(rows)
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].apply(lambda s: ILLEGAL_CHARACTERS_RE.sub('', s) if isinstance(s, str) else s)

# 4) Guardar a Excel
out = 'fichas_legislatura_mejorado.xlsx'
df.to_excel(out, index=False)
print("✅ Excel generado en:", out)


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

✅ Excel generado en: fichas_legislatura_mejorado.xlsx
